In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import tqdm
import numpy as np

from fugashi import Tagger
from linalgo.annotate import Sequence2SequenceTransformer
from linalgo.hub import BQClient

from wsd.models import JMDict


In [ ]:
# Fetch documents and annotations from BigQuery
task_id = os.getenv('LINHUB_TASK')
client = BQClient(task_id)

annotations = client.get_annotations()
docs = client.get_documents()
docs = [doc for doc in docs if len(doc.annotations) > 0]


In [ ]:
# Use the baseline `JMDict` model to disambiguate.
jmdict = JMDict()

y_pred = []
for doc in tqdm.tqdm(docs):
    yp = jmdict.predict(doc.content)
    y_pred.extend(yp)
y_pred = np.array(y_pred)


In [ ]:
# Retrieve the ground truth labels
tagger = Tagger('-Owakati')

def tokenize(text):
    idx = 0
    for token in tagger(text):
        yield idx, token.surface
        idx += len(token.surface)
    
transformer = Sequence2SequenceTransformer(tokenize_fn=tokenize)
input_sequence, output_sequence = transformer.transform(docs)
y_true = np.array([yt for seq in output_sequence for yt in seq])

In [ ]:
# Compute accuracy
acc = np.sum(y_true == y_pred) / len(y_true)
print(f"accuracy = {acc:%}")